In [ ]:
# Keep here all the needed functions



In [ ]:
# Status constants
VALID_STATUSES = {'START', 'ON', 'STOP'}

# Required columns
REQUIRED_COLUMNS = {'production_line_id', 'status', 'timestamp'}

In [ ]:
# Υπολογίζει τη διάρκεια, Επιστρέφει το dictionary
def _make_session(line_id, start, stop, idx):
    if stop is not None:
        duration = stop - start
        duration_sec = duration.total_seconds()
    else:
        duration = pd.NaT
        duration_sec = float('nan')
    return {
        'production_line_id': line_id,
        'start_timestamp':    start,
        'stop_timestamp':     stop if stop is not None else pd.NaT,
        'duration':           duration,
        'duration_seconds':   duration_sec,
        'session_index':      idx,
    }

In [ ]:
#Διαβάζει ένα CSV από το Lakehouse, καθαρίζει τα δεδομένα , ταξινομεί ανά γραμμή και χρόνο, και επιστρέφει έτοιμο DataFrame.
def load_events(csv_path):
    """
    Load and validate raw production events from the Lakehouse Files section.
    """

    # ── Read CSV ─────────────────────────────────────────────────────────────
    df = pd.read_csv(csv_path)
    print(f'[loader] Read {len(df)} rows from {csv_path}')

    # ── 1. Column check ──────────────────────────────────────────────────────
    missing = REQUIRED_COLUMNS - set(df.columns)
    if missing:
        raise ValueError(f'Input is missing required columns: {missing}')

    df = df[list(REQUIRED_COLUMNS)].copy()

    df['production_line_id'] = df['production_line_id'].astype(str).str.strip()
    df['status']             = df['status'].astype(str).str.strip().str.upper()
    df['timestamp']          = pd.to_datetime(df['timestamp'], utc=False)

    df = df.sort_values(['production_line_id', 'timestamp']).reset_index(drop=True)
    return df

In [ ]:
#Κρατά μόνο τα START/STOP events (πετά τα ON), και για κάθε γραμμή παραγωγής τα διατρέχει
#χρονολογικά ζευγαρώνοντας κάθε START με το επόμενο STOP του. 
#Επιστρέφει DataFrame με μία γραμμή ανά session.
def build_sessions(events):
    """
    Pair START and STOP events into sessions for every production line.

    Returns a DataFrame with one row per session:
    production_line_id, start_timestamp, stop_timestamp,
    duration, duration_seconds, session_index.
    """
    boundary = events[events['status'].isin(['START', 'STOP'])].copy()
    boundary = boundary.sort_values(['production_line_id', 'timestamp'])
    sessions = []

    for line_id, grp in boundary.groupby('production_line_id', sort=True):
        grp = grp.reset_index(drop=True)
        open_start = None
        session_idx = 0

        for _, row in grp.iterrows():
            status = row['status']
            ts     = row['timestamp']

            if status == 'START':
                if open_start is not None:
                    
                    session_idx += 1
                    sessions.append(_make_session(line_id, open_start, None, session_idx))
                open_start = ts

            elif status == 'STOP':
                if open_start is None:
                 
                    continue
                session_idx += 1
                sessions.append(_make_session(line_id, open_start, ts, session_idx))
                open_start = None

        if open_start is not None:
            session_idx += 1
            sessions.append(_make_session(line_id, open_start, None, session_idx))

    if not sessions:
        return pd.DataFrame(columns=['production_line_id','start_timestamp','stop_timestamp','duration','duration_seconds','session_index'])

    df = pd.DataFrame(sessions)
    return df.sort_values(['production_line_id', 'start_timestamp']).reset_index(drop=True)

In [ ]:
def line_sessions(events, production_line_id):
    """
    Return the session table for a specific production line.
    
    """
    available = events['production_line_id'].unique().tolist()
    if production_line_id not in available:
        raise ValueError(f'Line {production_line_id} not found. Available: {sorted(available)}')

    sessions = build_sessions(events)
    result = sessions[sessions['production_line_id'] == production_line_id].copy()
    result['duration_hms'] = result['duration'].apply(_fmt_duration)
    return result[['start_timestamp', 'stop_timestamp', 'duration', 'duration_hms']].reset_index(drop=True)

In [ ]:
def _sec_to_timedelta(seconds):
    return pd.Timedelta(seconds=seconds)
#converts duration → readable text like 01h 05m 10s
def _fmt_duration(td):
    if pd.isna(td):
        return 'open (no STOP)'
    total_sec = int(td.total_seconds())
    h, remainder = divmod(abs(total_sec), 3600)
    m, s = divmod(remainder, 60)
    return f'{h:02d}h {m:02d}m {s:02d}s'

In [ ]:
#Ρυθμίζει το pandas να εμφανίζει όλες τις στήλες και τυπώνει το DataFrame ολόκληρο σε μορφή πίνακα.

def _print_df(df):
    pd.set_option('display.max_columns', None)
    pd.set_option('display.width', 120)
    pd.set_option('display.max_colwidth', 30)
    print(df.to_string(index=True))
    print()

In [ ]:
def floor_uptime_downtime(events):
    """
    Compute total uptime and downtime per line, plus a FLOOR TOTAL row.
    Uptime  = sum of session durations.
    Downtime = sum of gaps between consecutive sessions.
    """
    sessions = build_sessions(events)
    rows = []

    for line_id, grp in sessions.groupby('production_line_id', sort=True):
        grp = grp.sort_values('start_timestamp').reset_index(drop=True)
        uptime_sec   = grp['duration_seconds'].dropna().sum()
        downtime_sec = 0.0

        for i in range(1, len(grp)):
            prev_stop  = grp.loc[i - 1, 'stop_timestamp']
            curr_start = grp.loc[i, 'start_timestamp']
            if pd.notna(prev_stop) and pd.notna(curr_start):
                gap = (curr_start - prev_stop).total_seconds()
                if gap > 0:
                    downtime_sec += gap

        rows.append({
            'production_line_id': line_id,
            'total_uptime':       _sec_to_timedelta(uptime_sec),
            'total_downtime':     _sec_to_timedelta(downtime_sec),
            'uptime_seconds':     uptime_sec,
            'downtime_seconds':   downtime_sec,
        })

    df = pd.DataFrame(rows)
    total_up   = df['uptime_seconds'].sum()
    total_down = df['downtime_seconds'].sum()
    total_row  = pd.DataFrame([{
        'production_line_id': 'FLOOR TOTAL',
        'total_uptime':       _sec_to_timedelta(total_up),
        'total_downtime':     _sec_to_timedelta(total_down),
        'uptime_seconds':     total_up,
        'downtime_seconds':   total_down,
    }])
    return pd.concat([df, total_row], ignore_index=True)

In [ ]:
def most_downtime_line(events):
    """
    Return the production line with the highest total downtime.
    """
    summary    = floor_uptime_downtime(events)
    lines_only = summary[summary['production_line_id'] != 'FLOOR TOTAL'].copy()
    if lines_only.empty:
        raise ValueError('No session data available.')
    #επιστρέφει τον index της γραμμής με το μέγιστο downtime, και πα΄ρνει όλη την γραμμή
    worst = lines_only.loc[lines_only['downtime_seconds'].idxmax()]
    return {
        'production_line_id': worst['production_line_id'],
        'total_downtime':     worst['total_downtime'],
        'downtime_seconds':   worst['downtime_seconds'],
        'downtime_hms':       _fmt_duration(worst['total_downtime']),
    }

In [ ]:
def print_report(events, line_id='gr-np-47'):
    """
    Print a full KPI report covering BQ1, BQ2, and BQ3.
    """
    # BQ1
   
    print(f'  BQ1 - Session table for: {line_id}')
   
    _print_df(line_sessions(events, line_id))

    # BQ2
    
    print('  BQ2 - Total uptime / downtime per line')
  
    summary = floor_uptime_downtime(events)
    display = summary[['production_line_id','total_uptime','total_downtime','uptime_seconds','downtime_seconds']].copy()
    display['total_uptime']     = display['total_uptime'].apply(_fmt_duration)
    display['total_downtime']   = display['total_downtime'].apply(_fmt_duration)
    display['uptime_seconds']   = display['uptime_seconds'].apply(lambda x: f'{x:,.0f}')
    display['downtime_seconds'] = display['downtime_seconds'].apply(lambda x: f'{x:,.0f}')
    _print_df(display)

    # BQ3
    
    print('  BQ3 - Line with the most downtime')
   
    worst = most_downtime_line(events)
    print(f"  Line ID       : {worst['production_line_id']}")
    print(f"  Total downtime: {worst['downtime_hms']}")
    print(f"  (seconds)     : {worst['downtime_seconds']:,.0f} s")
